##Jai Ganesha

# 03 — COCO 2017 Preprocessing

Goal: Filter COCO 2017 to only the classes that overlap with VisDrone
(person, bicycle, car, motorcycle, bus, truck), convert to YOLO format,
and save a coco_filtered.yaml for optional YOLO-World pretraining augmentation.

Inputs  : Raw COCO 2017 (instances_train2017.json, instances_val2017.json)
Outputs :
  - processed/coco/images/{train,val}/      (symlinks — no copying, saves Drive space)
  - processed/coco/labels/{train,val}/      (YOLO .txt label files)
  - processed/coco/coco_filtered.yaml
  - processed/coco/class_dist.csv
  - processed/coco/coco_to_yolo_map.json

Why filter? COCO has 80 classes. We only want the 6 classes that are
semantically meaningful for drone-view urban analytics, matching VisDrone.

COCO annotation format (JSON):
  bbox: [x_min, y_min, width, height]  (pixel absolute)
  → convert to YOLO: x_center_norm, y_center_norm, width_norm, height_norm

Class mapping (COCO category_id → our YOLO class_id):
  person       (1)  → 0
  bicycle      (2)  → 1
  car          (3)  → 2
  motorcycle   (4)  → 3
  bus          (6)  → 4
  truck        (8)  → 5

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Imports and config load
# ─────────────────────────────────────────────────────────────

import json, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import cv2
import yaml

# ── Load config ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
CONFIG_PATH = "/content/drive/MyDrive/DLCV_OV_Analytics/configs/config.json"
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

RAW_COCO   = cfg["raw_coco"]
PROC_COCO  = cfg["proc_coco"]
METRICS_DIR = cfg["metrics_dir"]
VIZ_DIR    = cfg["viz_dir"]

# ── Output directories ────────────────────────────────────────
PROC_COCO_IMG_TRAIN = os.path.join(PROC_COCO, "images", "train")
PROC_COCO_IMG_VAL   = os.path.join(PROC_COCO, "images", "val")
PROC_COCO_LBL_TRAIN = os.path.join(PROC_COCO, "labels", "train")
PROC_COCO_LBL_VAL   = os.path.join(PROC_COCO, "labels", "val")

for d in [PROC_COCO_IMG_TRAIN, PROC_COCO_IMG_VAL,
          PROC_COCO_LBL_TRAIN, PROC_COCO_LBL_VAL]:
    os.makedirs(d, exist_ok=True)

# ── Classes: COCO category names we want → our YOLO class IDs ─
# Only these 6 classes overlap meaningfully with VisDrone
SELECTED_CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

# Our YOLO class IDs (0-based)
YOLO_CLASS_ID = {name: idx for idx, name in enumerate(SELECTED_CLASSES)}

print("✅ Setup complete.")
print(f"   RAW COCO  : {RAW_COCO}")
print(f"   PROC COCO : {PROC_COCO}")
print(f"   Selected classes ({len(SELECTED_CLASSES)}): {SELECTED_CLASSES}")
print(f"   YOLO class IDs: {YOLO_CLASS_ID}")

Mounted at /content/drive
✅ Setup complete.
   RAW COCO  : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO
   PROC COCO : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco
   Selected classes (6): ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
   YOLO class IDs: {'person': 0, 'bicycle': 1, 'car': 2, 'motorcycle': 3, 'bus': 4, 'truck': 5}


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Load COCO JSON and build category → YOLO ID mapping
# ─────────────────────────────────────────────────────────────

def build_coco_category_map(ann_json_path, selected_classes):
    """
    Load COCO annotation JSON and return:
      - coco_cat_to_yolo: dict mapping COCO category_id → our YOLO class_id
      - image_id_to_info: dict mapping image_id → {file_name, width, height}
      - filtered_anns: list of annotations for selected classes only
    """
    print(f"  Loading {Path(ann_json_path).name}...")
    with open(ann_json_path, "r") as f:
        data = json.load(f)

    # Build COCO category_name → category_id lookup
    name_to_coco_id = {c["name"]: c["id"] for c in data["categories"]}

    # Map COCO category_id → our YOLO class_id (only for selected classes)
    coco_cat_to_yolo = {}
    for cls_name in selected_classes:
        if cls_name not in name_to_coco_id:
            print(f"  ⚠️  Class '{cls_name}' not found in COCO categories — skipping")
            continue
        coco_id  = name_to_coco_id[cls_name]
        yolo_id  = YOLO_CLASS_ID[cls_name]
        coco_cat_to_yolo[coco_id] = yolo_id
        print(f"    COCO '{cls_name}' (id={coco_id}) → YOLO class {yolo_id}")

    # Build image_id → file info lookup
    image_id_to_info = {
        img["id"]: {
            "file_name": img["file_name"],
            "width"    : img["width"],
            "height"   : img["height"],
        }
        for img in data["images"]
    }

    # Filter annotations to selected classes only
    selected_coco_ids = set(coco_cat_to_yolo.keys())
    filtered_anns = [
        ann for ann in data["annotations"]
        if ann["category_id"] in selected_coco_ids
    ]

    print(f"  Total annotations    : {len(data['annotations']):,}")
    print(f"  Filtered annotations : {len(filtered_anns):,} "
          f"({100*len(filtered_anns)/len(data['annotations']):.1f}%)")
    print(f"  Total images         : {len(image_id_to_info):,}")

    del data
    return coco_cat_to_yolo, image_id_to_info, filtered_anns


# ── Run for train split ───────────────────────────────────────
print("=" * 50)
print("  Train split")
print("=" * 50)
train_ann_path = os.path.join(RAW_COCO, "annotations", "instances_train2017.json")
train_cat_map, train_img_info, train_anns = build_coco_category_map(
    train_ann_path, SELECTED_CLASSES
)

# ── Run for val split ─────────────────────────────────────────
print("\n" + "=" * 50)
print("  Val split")
print("=" * 50)
val_ann_path = os.path.join(RAW_COCO, "annotations", "instances_val2017.json")
val_cat_map, val_img_info, val_anns = build_coco_category_map(
    val_ann_path, SELECTED_CLASSES
)

# ── Save the mapping for reference ───────────────────────────
map_save_path = os.path.join(PROC_COCO, "coco_to_yolo_map.json")
os.makedirs(PROC_COCO, exist_ok=True)
with open(map_save_path, "w") as f:
    json.dump({
        "selected_classes"  : SELECTED_CLASSES,
        "yolo_class_ids"    : YOLO_CLASS_ID,
        "coco_cat_to_yolo"  : {str(k): v for k, v in train_cat_map.items()},
    }, f, indent=2)
print(f"\n✅ Category map saved: {map_save_path}")

  Train split
  Loading instances_train2017.json...
    COCO 'person' (id=1) → YOLO class 0
    COCO 'bicycle' (id=2) → YOLO class 1
    COCO 'car' (id=3) → YOLO class 2
    COCO 'motorcycle' (id=4) → YOLO class 3
    COCO 'bus' (id=6) → YOLO class 4
    COCO 'truck' (id=8) → YOLO class 5
  Total annotations    : 860,001
  Filtered annotations : 338,212 (39.3%)
  Total images         : 118,287

  Val split
  Loading instances_val2017.json...
    COCO 'person' (id=1) → YOLO class 0
    COCO 'bicycle' (id=2) → YOLO class 1
    COCO 'car' (id=3) → YOLO class 2
    COCO 'motorcycle' (id=4) → YOLO class 3
    COCO 'bus' (id=6) → YOLO class 4
    COCO 'truck' (id=8) → YOLO class 5
  Total annotations    : 36,781
  Filtered annotations : 14,323 (38.9%)
  Total images         : 5,000

✅ Category map saved: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/coco_to_yolo_map.json


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Core COCO → YOLO label conversion function
# ─────────────────────────────────────────────────────────────

def coco_anns_to_yolo(annotations, image_id_to_info, coco_cat_to_yolo):
    """
    Group annotations by image and convert each to YOLO format.

    COCO bbox: [x_min, y_min, width, height]  (absolute pixels)
    YOLO line: class_id x_center_norm y_center_norm width_norm height_norm

    Returns:
        image_labels: dict {image_id: [yolo_line, ...]}
        stats: conversion statistics
    """
    # Group annotations by image_id
    anns_by_image = defaultdict(list)
    for ann in annotations:
        anns_by_image[ann["image_id"]].append(ann)

    image_labels = {}
    stats = {
        "n_images_with_anns": 0,
        "n_boxes"           : 0,
        "n_skipped_degen"   : 0,
        "n_skipped_oob"     : 0,
        "n_small"           : 0,
        "class_counts"      : defaultdict(int),
    }

    for image_id, anns in anns_by_image.items():
        if image_id not in image_id_to_info:
            continue

        info   = image_id_to_info[image_id]
        img_w  = info["width"]
        img_h  = info["height"]
        lines  = []

        for ann in anns:
            coco_cat_id = ann["category_id"]
            if coco_cat_id not in coco_cat_to_yolo:
                continue

            x, y, w, h = ann["bbox"]  # COCO: [x_min, y_min, width, height]

            # Skip degenerate boxes
            if w <= 0 or h <= 0:
                stats["n_skipped_degen"] += 1
                continue

            # Clamp to image boundaries
            x = max(0.0, min(x, img_w - 1))
            y = max(0.0, min(y, img_h - 1))
            w = min(w, img_w - x)
            h = min(h, img_h - y)

            if w <= 0 or h <= 0:
                stats["n_skipped_oob"] += 1
                continue

            # Convert to YOLO normalized center format
            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            w_norm   = w / img_w
            h_norm   = h / img_h

            # Final [0,1] clamp
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            w_norm   = max(0.0, min(1.0, w_norm))
            h_norm   = max(0.0, min(1.0, h_norm))

            yolo_cls = coco_cat_to_yolo[coco_cat_id]
            stats["class_counts"][yolo_cls] += 1
            stats["n_boxes"] += 1

            if (w * h) < 1024:
                stats["n_small"] += 1

            lines.append(
                f"{yolo_cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            )

        if lines:
            image_labels[image_id] = lines
            stats["n_images_with_anns"] += 1

    return image_labels, stats


# ── Run for train and val ─────────────────────────────────────
print("Converting train annotations...")
train_labels, train_stats = coco_anns_to_yolo(
    train_anns, train_img_info, train_cat_map
)
print(f"  ✅ {train_stats['n_images_with_anns']:,} images with annotations")
print(f"     YOLO boxes    : {train_stats['n_boxes']:,}")
print(f"     Skipped degen : {train_stats['n_skipped_degen']:,}")
print(f"     Skipped oob   : {train_stats['n_skipped_oob']:,}")
print(f"     Small objects : {train_stats['n_small']:,}")

print("\nConverting val annotations...")
val_labels, val_stats = coco_anns_to_yolo(
    val_anns, val_img_info, val_cat_map
)
print(f"  ✅ {val_stats['n_images_with_anns']:,} images with annotations")
print(f"     YOLO boxes    : {val_stats['n_boxes']:,}")
print(f"     Skipped degen : {val_stats['n_skipped_degen']:,}")
print(f"     Skipped oob   : {val_stats['n_skipped_oob']:,}")
print(f"     Small objects : {val_stats['n_small']:,}")

Converting train annotations...
  ✅ 70,082 images with annotations
     YOLO boxes    : 338,211
     Skipped degen : 1
     Skipped oob   : 0
     Small objects : 107,300

Converting val annotations...
  ✅ 2,968 images with annotations
     YOLO boxes    : 14,323
     Skipped degen : 0
     Skipped oob   : 0
     Small objects : 4,471


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 (FINAL FIX): Write labels locally first, then copy to Drive
# Writing 70K files directly to Drive triggers OSError [Errno 5]
# (Drive API rate limit). Writing locally first avoids this entirely.
# ─────────────────────────────────────────────────────────────

import os, shutil
from pathlib import Path
from tqdm import tqdm

# ── Local staging dirs on Colab's SSD (fast, no rate limits) ─
LOCAL_STAGE      = "/content/staging/coco"
LOCAL_LBL_TRAIN  = os.path.join(LOCAL_STAGE, "labels/train")
LOCAL_LBL_VAL    = os.path.join(LOCAL_STAGE, "labels/val")

os.makedirs(LOCAL_LBL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_LBL_VAL,   exist_ok=True)

def write_labels_local(image_labels, image_id_to_info, lbl_out_dir, split_name):
    n_written = 0
    for image_id, yolo_lines in tqdm(image_labels.items(), desc=f"{split_name} (local)"):
        file_name = image_id_to_info[image_id]["file_name"]
        dst_lbl   = os.path.join(lbl_out_dir, Path(file_name).stem + ".txt")
        with open(dst_lbl, "w") as f:
            f.write("\n".join(yolo_lines))
        n_written += 1
    return n_written

# ── Step 1: Write to local SSD ────────────────────────────────
print("Step 1: Writing labels to local Colab SSD...")
n_train = write_labels_local(train_labels, train_img_info, LOCAL_LBL_TRAIN, "train")
print(f"  ✅ Train: {n_train:,} label files written locally")

n_val = write_labels_local(val_labels, val_img_info, LOCAL_LBL_VAL, "val")
print(f"  ✅ Val  : {n_val:,} label files written locally")

# ── Step 2: Copy entire folder to Drive in one operation ──────
print("\nStep 2: Copying labels folder to Drive (one bulk transfer)...")

DRIVE_LBL = os.path.join(PROC_COCO, "labels")
if os.path.exists(DRIVE_LBL):
    shutil.rmtree(DRIVE_LBL)   # remove old/partial labels folder

shutil.copytree(
    os.path.join(LOCAL_STAGE, "labels"),
    DRIVE_LBL
)
print(f"  ✅ Labels copied to Drive: {DRIVE_LBL}")

# ── Step 3: Verify ────────────────────────────────────────────
print("\nVerifying Drive label counts...")
for split, lbl_dir in [("train", PROC_COCO_LBL_TRAIN), ("val", PROC_COCO_LBL_VAL)]:
    n = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f"  {split}: {n:,} label files on Drive")

print("\n✅ Cell 4 complete.")

Step 1: Writing labels to local Colab SSD...


train (local): 100%|██████████| 70082/70082 [00:03<00:00, 19266.94it/s]


  ✅ Train: 70,082 label files written locally


val (local): 100%|██████████| 2968/2968 [00:00<00:00, 19879.74it/s]


  ✅ Val  : 2,968 label files written locally

Step 2: Copying labels folder to Drive (one bulk transfer)...
  ✅ Labels copied to Drive: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/labels

Verifying Drive label counts...
  train: 70,082 label files on Drive
  val: 2,968 label files on Drive

✅ Cell 4 complete.


## Cell 4b — Recovery Cell (Run ONLY if Cell 4 was interrupted)
 Use this if Drive disconnected mid-write and labels/train is incomplete.
Check with: len(os.listdir(PROC_COCO_LBL_TRAIN)) — expected 70,082

This cell is safe to skip during a clean run.

In [ ]:
import os, shutil, json, yaml
import pandas as pd
from pathlib import Path
from tqdm import tqdm

PROC_COCO       = "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco"
PROC_COCO_LBL_TRAIN = os.path.join(PROC_COCO, "labels/train")
PROC_COCO_LBL_VAL   = os.path.join(PROC_COCO, "labels/val")
RAW_COCO        = "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO"
SELECTED_CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

LOCAL_LBL_TRAIN = "/content/staging/coco/labels/train"
os.makedirs(LOCAL_LBL_TRAIN, exist_ok=True)

# ── Write all 70,082 train labels locally first ───────────────
print("Writing train labels to local SSD...")
n_written = 0
for image_id, yolo_lines in tqdm(train_labels.items(), desc="train"):
    file_name = train_img_info[image_id]["file_name"]
    dst = os.path.join(LOCAL_LBL_TRAIN, Path(file_name).stem + ".txt")
    with open(dst, "w") as f:
        f.write("\n".join(yolo_lines))
    n_written += 1
print(f"✅ {n_written:,} train label files written locally")

# ── Clear incomplete Drive folder and replace with full set ───
print("\nReplacing incomplete Drive labels/train folder...")
if os.path.exists(PROC_COCO_LBL_TRAIN):
    shutil.rmtree(PROC_COCO_LBL_TRAIN)

shutil.copytree(LOCAL_LBL_TRAIN, PROC_COCO_LBL_TRAIN)
print(f"✅ labels/train copied to Drive")

# ── Verify ────────────────────────────────────────────────────
n_train = len(os.listdir(PROC_COCO_LBL_TRAIN))
n_val   = len(os.listdir(PROC_COCO_LBL_VAL))
print(f"\ntrain labels on Drive : {n_train:,}  (expected 70,082)")
print(f"val   labels on Drive : {n_val:,}  (expected 2,968)")

# ── Write YAML + CSV + JSON ───────────────────────────────────
LOCAL_META = "/content/staging/coco"

yaml_content = {
    "path"        : PROC_COCO,
    "train"       : os.path.join(RAW_COCO, "train2017"),
    "val"         : os.path.join(RAW_COCO, "val2017"),
    "train_labels": "labels/train",
    "val_labels"  : "labels/val",
    "nc"          : len(SELECTED_CLASSES),
    "names"       : SELECTED_CLASSES,
}
with open(os.path.join(LOCAL_META, "coco_filtered.yaml"), "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

rows = []
for split_name, stats in [("train", train_stats), ("val", val_stats)]:
    for cls_id, count in stats["class_counts"].items():
        rows.append({"split": split_name, "class_id": cls_id,
                     "class_name": SELECTED_CLASSES[cls_id], "count": count})
pd.DataFrame(rows).to_csv(os.path.join(LOCAL_META, "class_dist.csv"), index=False)

coco_to_yolo_map = {str(k): {"yolo_id": v, "name": SELECTED_CLASSES[v]}
                    for k, v in {1:0, 2:1, 3:2, 4:3, 6:4, 8:5}.items()}
with open(os.path.join(LOCAL_META, "coco_to_yolo_map.json"), "w") as f:
    json.dump(coco_to_yolo_map, f, indent=2)

for fname in ["coco_filtered.yaml", "class_dist.csv", "coco_to_yolo_map.json"]:
    shutil.copy2(os.path.join(LOCAL_META, fname), PROC_COCO)
    size_kb = os.path.getsize(os.path.join(PROC_COCO, fname)) / 1024
    print(f"✅ {fname} ({size_kb:.1f} KB)")

print("\n✅ Notebook 03 recovery complete.")

Writing train labels to local SSD...


train: 100%|██████████| 70082/70082 [00:03<00:00, 18821.79it/s]


✅ 70,082 train label files written locally

Replacing incomplete Drive labels/train folder...
✅ labels/train copied to Drive

train labels on Drive : 70,082  (expected 70,082)
val   labels on Drive : 2,968  (expected 2,968)
✅ coco_filtered.yaml (0.3 KB)
✅ class_dist.csv (0.2 KB)
✅ coco_to_yolo_map.json (0.3 KB)

✅ Notebook 03 recovery complete.


In [ ]:
# # Spot-check one label file
# import os, random
# lbl_dir = "/content/staging/coco/labels/train"
# sample = random.choice(os.listdir(lbl_dir))
# with open(os.path.join(lbl_dir, sample)) as f:
#     print(f.read())
# # Should show lines like: 0 0.512345 0.423456 0.123456 0.098765
# # NOT: 0 0.512345...\\n0 0.423456...  (all on one line)

0 0.446797 0.773033 0.131937 0.426956


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: Save YAML, class distribution CSV, and COCO→YOLO map
# ─────────────────────────────────────────────────────────────

import yaml, json, os, shutil
import pandas as pd

# ── Define constants (should match what was used in Cell 3) ───
SELECTED_CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

COCO_TO_YOLO_ID = {
    1: 0,   # person
    2: 1,   # bicycle
    3: 2,   # car
    4: 3,   # motorcycle
    6: 4,   # bus
    8: 5,   # truck
}

# ── Build YAML content ────────────────────────────────────────
yaml_content = {
    "path"         : PROC_COCO,
    "train"        : os.path.join(RAW_COCO, "train2017"),
    "val"          : os.path.join(RAW_COCO, "val2017"),
    "train_labels" : "labels/train",
    "val_labels"   : "labels/val",
    "nc"           : len(SELECTED_CLASSES),
    "names"        : SELECTED_CLASSES,
}

# ── Build class distribution DataFrame ───────────────────────
rows = []
for split_name, stats in [("train", train_stats), ("val", val_stats)]:
    for cls_id, count in stats["class_counts"].items():
        rows.append({
            "split"      : split_name,
            "class_id"   : cls_id,
            "class_name" : SELECTED_CLASSES[cls_id],
            "count"      : count,
        })

df_dist = pd.DataFrame(rows)
pivot   = df_dist.pivot_table(
    index="class_name", columns="split",
    values="count", aggfunc="sum", fill_value=0
)

# ── Build COCO→YOLO class ID map ─────────────────────────────
coco_to_yolo_map = {
    str(coco_id): {"yolo_id": yolo_id, "name": SELECTED_CLASSES[yolo_id]}
    for coco_id, yolo_id in COCO_TO_YOLO_ID.items()
}

# ── Write locally first, then copy to Drive ───────────────────
LOCAL_META = "/content/staging/coco"
os.makedirs(LOCAL_META, exist_ok=True)

local_yaml_path = os.path.join(LOCAL_META, "coco_filtered.yaml")
local_dist_path = os.path.join(LOCAL_META, "class_dist.csv")
local_map_path  = os.path.join(LOCAL_META, "coco_to_yolo_map.json")

with open(local_yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

df_dist.to_csv(local_dist_path, index=False)

with open(local_map_path, "w") as f:
    json.dump(coco_to_yolo_map, f, indent=2)

for local_f in [local_yaml_path, local_dist_path, local_map_path]:
    shutil.copy2(local_f, PROC_COCO)

yaml_path     = os.path.join(PROC_COCO, "coco_filtered.yaml")
dist_path     = os.path.join(PROC_COCO, "class_dist.csv")
map_save_path = os.path.join(PROC_COCO, "coco_to_yolo_map.json")

print(f"✅ coco_filtered.yaml saved : {yaml_path}")
print(f"✅ class_dist.csv saved     : {dist_path}")
print(f"✅ coco_to_yolo_map.json    : {map_save_path}")
print(f"\nFiltered COCO — Class Distribution:")
print(pivot.to_string())

✅ coco_filtered.yaml saved : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/coco_filtered.yaml
✅ class_dist.csv saved     : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/class_dist.csv
✅ coco_to_yolo_map.json    : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/coco_to_yolo_map.json

Filtered COCO — Class Distribution:
split        train    val
class_name               
bicycle       7113    316
bus           6069    285
car          43867   1932
motorcycle    8725    371
person      262464  11004
truck         9973    415


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6: Visualize sample COCO filtered annotations
# ─────────────────────────────────────────────────────────────

COCO_COLOR_MAP = {
    0: (255, 80,  80),   # person     — red
    1: (80,  180, 80),   # bicycle    — green
    2: (80,  120, 255),  # car        — blue
    3: (255, 160, 80),   # motorcycle — orange
    4: (255, 100, 180),  # bus        — pink
    5: (255, 220, 60),   # truck      — yellow
}

def draw_yolo_on_image(img_path, lbl_path, class_names, color_map):
    """Load image and draw YOLO bounding boxes on it."""
    img = cv2.imread(img_path)
    if img is None:
        return None, 0
    img    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w   = img.shape[:2]
    n_boxes = 0

    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)
                color = color_map.get(cls_id, (200, 200, 200))
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                cv2.putText(img, label, (x1, max(0, y1 - 4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
                n_boxes += 1
    return img, n_boxes

# ── Sample 6 val images that have labels ─────────────────────
lbl_files = sorted(os.listdir(PROC_COCO_LBL_VAL))[:20]
samples   = []
for lbl_file in lbl_files:
    img_file = Path(lbl_file).stem + ".jpg"
    img_path = os.path.join(PROC_COCO_IMG_VAL, img_file)
    lbl_path = os.path.join(PROC_COCO_LBL_VAL, lbl_file)
    if os.path.exists(img_path):
        samples.append((img_path, lbl_path))
    if len(samples) == 6:
        break

if not samples:
    print("⚠️  No val images found via symlinks. Skipping visualization.")
    print("   Labels were written correctly — this is a Drive symlink read issue.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, (img_path, lbl_path) in enumerate(samples):
        img, n_boxes = draw_yolo_on_image(img_path, lbl_path,
                                          SELECTED_CLASSES, COCO_COLOR_MAP)
        if img is not None:
            axes[i].imshow(img)
            axes[i].set_title(f"{Path(img_path).name}\n{n_boxes} boxes", fontsize=8)
        axes[i].axis("off")

    plt.suptitle("COCO Filtered — YOLO Labels Verification (Val)", fontsize=13)
    plt.tight_layout()
    save_path = os.path.join(VIZ_DIR, "coco_filtered_labels_check.png")
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")

⚠️  No val images found via symlinks. Skipping visualization.
   Labels were written correctly — this is a Drive symlink read issue.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7: Final summary
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("  COCO PREPROCESSING SUMMARY")
print("=" * 60)

for split_name, stats, lbl_dir, raw_img_dir in [
    ("train", train_stats, PROC_COCO_LBL_TRAIN, os.path.join(RAW_COCO, "train2017")),
    ("val",   val_stats,   PROC_COCO_LBL_VAL,   os.path.join(RAW_COCO, "val2017")),
]:
    n_lbls = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f"\n  Split: {split_name}")
    print(f"    Label files written  : {n_lbls:,}")
    print(f"    Total YOLO boxes     : {stats['n_boxes']:,}")
    print(f"    Small objects        : {stats['n_small']:,}  "
          f"({100*stats['n_small']/max(stats['n_boxes'],1):.1f}% of boxes)")
    print(f"    Skipped (degen/oob)  : "
          f"{stats['n_skipped_degen']:,} / {stats['n_skipped_oob']:,}")
    top3 = sorted(stats["class_counts"].items(), key=lambda x: -x[1])[:3]
    print(f"    Top 3 classes        : "
          f"{[(SELECTED_CLASSES[k], v) for k, v in top3]}")
    print(f"    Raw images at        : {raw_img_dir}")

print(f"\n  Output files:")
print(f"    coco_filtered.yaml : {yaml_path}")
print(f"    class_dist.csv     : {dist_path}")
print(f"    coco_to_yolo_map   : {map_save_path}")
print("\n✅ Notebook 03 complete. Ready for Notebook 04.")

  COCO PREPROCESSING SUMMARY

  Split: train
    Label files written  : 70,082
    Total YOLO boxes     : 338,211
    Small objects        : 107,300  (31.7% of boxes)
    Skipped (degen/oob)  : 1 / 0
    Top 3 classes        : [('person', 262464), ('car', 43867), ('truck', 9973)]
    Raw images at        : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO/train2017

  Split: val
    Label files written  : 2,968
    Total YOLO boxes     : 14,323
    Small objects        : 4,471  (31.2% of boxes)
    Skipped (degen/oob)  : 0 / 0
    Top 3 classes        : [('person', 11004), ('car', 1932), ('truck', 415)]
    Raw images at        : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO/val2017

  Output files:
    coco_filtered.yaml : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/coco_filtered.yaml
    class_dist.csv     : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/class_dist.csv
    coco_to_yolo_map   : /content/drive/MyDrive/DLCV_OV

In [ ]:
# ─────────────────────────────────────────────────────────────
# Force Drive sync — flush all pending writes
# ─────────────────────────────────────────────────────────────
import subprocess, os

# Force OS-level flush
subprocess.run(["sync"], check=True)

# Verify all expected COCO output files exist
expected_files = [
    os.path.join(PROC_COCO, "coco_filtered.yaml"),
    os.path.join(PROC_COCO, "class_dist.csv"),
    os.path.join(PROC_COCO, "coco_to_yolo_map.json"),
]

print("Verifying COCO output files on Drive:")
all_ok = True
for fpath in expected_files:
    exists  = os.path.exists(fpath)
    size_kb = os.path.getsize(fpath) / 1024 if exists else 0
    status  = f"✅ ({size_kb:.1f} KB)" if exists else "❌ MISSING"
    print(f"   {status}  {os.path.basename(fpath)}")
    if not exists:
        all_ok = False

# Also verify label counts
print(f"\nLabel file counts:")
for split, lbl_dir in [("train", PROC_COCO_LBL_TRAIN), ("val", PROC_COCO_LBL_VAL)]:
    n = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f"   {split}: {n:,} label files")

if all_ok:
    print("\n✅ All files confirmed on Drive.")
else:
    print("\n⚠️  Some files missing — re-running Cell 5 will fix this.")

Verifying COCO output files on Drive:
   ✅ (0.3 KB)  coco_filtered.yaml
   ✅ (0.2 KB)  class_dist.csv
   ✅ (0.3 KB)  coco_to_yolo_map.json

Label file counts:
   train: 70,082 label files
   val: 2,968 label files

✅ All files confirmed on Drive.


In [ ]:
import os

for fname in ["coco_filtered.yaml", "class_dist.csv", "coco_to_yolo_map.json"]:
    fpath    = os.path.join(PROC_COCO, fname)
    exists   = os.path.exists(fpath)
    size_kb  = os.path.getsize(fpath) / 1024 if exists else 0
    print(f"  {'✅' if exists else '❌'} {fname}  ({size_kb:.1f} KB)")

# Print the Drive path — paste into Drive search bar
print(f"Search in Drive for: coco_filtered.yaml")
print(f"Full path: {os.path.join(PROC_COCO, 'coco_filtered.yaml')}")

  ✅ coco_filtered.yaml  (0.3 KB)
  ✅ class_dist.csv  (0.2 KB)
  ✅ coco_to_yolo_map.json  (0.3 KB)
Search in Drive for: coco_filtered.yaml
Full path: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/coco_filtered.yaml


In [ ]:
import os, json, yaml, pandas as pd

PROC_COCO = "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco"

# Read and print each file's actual content
print("=== coco_filtered.yaml ===")
with open(os.path.join(PROC_COCO, "coco_filtered.yaml")) as f:
    print(f.read())

print("\n=== class_dist.csv ===")
df = pd.read_csv(os.path.join(PROC_COCO, "class_dist.csv"))
print(df.to_string())

print("\n=== coco_to_yolo_map.json ===")
with open(os.path.join(PROC_COCO, "coco_to_yolo_map.json")) as f:
    print(json.dumps(json.load(f), indent=2))

print("\n=== Label counts ===")
for split in ["train", "val"]:
    lbl_dir = os.path.join(PROC_COCO, "labels", split)
    n = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f"  {split}: {n:,} label files")

=== coco_filtered.yaml ===
names:
- person
- bicycle
- car
- motorcycle
- bus
- truck
nc: 6
path: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco
train: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO/train2017
train_labels: labels/train
val: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO/val2017
val_labels: labels/val


=== class_dist.csv ===
    split  class_id  class_name   count
0   train         1     bicycle    7113
1   train         3  motorcycle    8725
2   train         0      person  262464
3   train         4         bus    6069
4   train         2         car   43867
5   train         5       truck    9973
6     val         1     bicycle     316
7     val         2         car    1932
8     val         0      person   11004
9     val         3  motorcycle     371
10    val         4         bus     285
11    val         5       truck     415

=== coco_to_yolo_map.json ===
{
  "1": {
    "yolo_id": 0,
    "name": "person"
  },
  "2": {

In [2]:
import os
PROC_COCO = "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco"

lbl_train = os.path.join(PROC_COCO, "labels/train")
lbl_val   = os.path.join(PROC_COCO, "labels/val")

n_train = len(os.listdir(lbl_train)) if os.path.isdir(lbl_train) else 0
n_val   = len(os.listdir(lbl_val))   if os.path.isdir(lbl_val)   else 0
print(f"train labels: {n_train:,}")
print(f"val   labels: {n_val:,}")

OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco/labels/train'

In [ ]:
import os, json, yaml, shutil, subprocess
import pandas as pd
from google.colab import drive

# ── Step 1: Hard remount ──────────────────────────────────────
subprocess.run("fusermount -u /content/drive", shell=True)
drive.mount('/content/drive', force_remount=True)

PROC_COCO        = "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco"
RAW_COCO         = "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO"
SELECTED_CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]
LOCAL_META       = "/content/staging/coco"
os.makedirs(LOCAL_META, exist_ok=True)

# ── Step 2: Write all 3 files locally first ───────────────────
yaml_content = {
    "path"        : PROC_COCO,
    "train"       : os.path.join(RAW_COCO, "train2017"),
    "val"         : os.path.join(RAW_COCO, "val2017"),
    "train_labels": "labels/train",
    "val_labels"  : "labels/val",
    "nc"          : len(SELECTED_CLASSES),
    "names"       : SELECTED_CLASSES,
}
with open(f"{LOCAL_META}/coco_filtered.yaml", "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

rows = []
for split_name, stats in [("train", train_stats), ("val", val_stats)]:
    for cls_id, count in stats["class_counts"].items():
        rows.append({"split": split_name, "class_id": cls_id,
                     "class_name": SELECTED_CLASSES[cls_id], "count": count})
pd.DataFrame(rows).to_csv(f"{LOCAL_META}/class_dist.csv", index=False)

coco_map = {str(k): {"yolo_id": v, "name": SELECTED_CLASSES[v]}
            for k, v in {1:0, 2:1, 3:2, 4:3, 6:4, 8:5}.items()}
with open(f"{LOCAL_META}/coco_to_yolo_map.json", "w") as f:
    json.dump(coco_map, f, indent=2)

print("✅ All 3 files written locally")

# ── Step 3: Use cp (OS-level) instead of shutil ──────────────
# cp uses OS-level I/O which flushes properly to FUSE mounts
for fname in ["coco_filtered.yaml", "class_dist.csv", "coco_to_yolo_map.json"]:
    src = f"{LOCAL_META}/{fname}"
    dst = f"{PROC_COCO}/{fname}"
    result = subprocess.run(f"cp '{src}' '{dst}' && sync", shell=True,
                            capture_output=True, text=True)
    if result.returncode == 0:
        # Read back immediately to confirm it's truly on Drive
        try:
            size = os.path.getsize(dst)
            with open(dst, "r") as f:
                content = f.read()
            print(f"✅ {fname} ({size} bytes) — read-back confirmed")
        except Exception as e:
            print(f"❌ {fname} write succeeded but read-back failed: {e}")
    else:
        print(f"❌ {fname} copy failed: {result.stderr}")

# ── Step 4: Force OS sync ─────────────────────────────────────
subprocess.run("sync", shell=True)
print("\n✅ sync complete — files are on Drive")

Mounted at /content/drive
✅ All 3 files written locally
✅ coco_filtered.yaml (334 bytes) — read-back confirmed
✅ class_dist.csv (256 bytes) — read-back confirmed
✅ coco_to_yolo_map.json (318 bytes) — read-back confirmed

✅ sync complete — files are on Drive
